# Banking intent classification: train on a free Colab GPU

Runs the TF-IDF baseline and fine-tunes DistilBERT on Banking77, then builds the results table.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`.
Full training takes roughly 5-10 minutes on a T4.

## 1. Get the code
Either clone your GitHub repo (edit the URL), **or** upload the project zip and unzip it.

In [ ]:
# Option A: clone from GitHub (replace with your repo URL)
!git clone https://github.com/Himani-gu/banking-intent-nlp.git
%cd banking-intent-nlp

# Option B: upload the zip instead
# from google.colab import files; files.upload()
# !unzip -q banking-intent-nlp.zip
# %cd banking-intent-nlp

In [ ]:
!pip install -q -e ".[dev]" "transformers>=4.40"   # Colab already ships a recent torch
!nvidia-smi | head -n 12

## 2. Sanity check (should pass in under a minute)

In [ ]:
!python -m pytest -q

## 3. Baselines at several training-set sizes
Full data first, then 10% / 25% / 50% for the data-efficiency curve.

In [ ]:
for frac in [1.0, 0.1, 0.25, 0.5]:
    !python -m intent_nlp.baseline --fraction {frac}

## 4. Fine-tune DistilBERT on the full training set
Best epoch is chosen on the validation split; the test set is used once at the end.

In [ ]:
!python -m intent_nlp.train_transformer --fp16

## 5. Data-efficiency runs (transformer with less data)
This is where pretrained models usually pull ahead of TF-IDF.

In [ ]:
for frac in [0.1, 0.25, 0.5]:
    !python -m intent_nlp.train_transformer --fp16 --fraction {frac}

## 6. Results table + figures

In [ ]:
!python scripts/make_report.py
from IPython.display import Image, display
display(Image('results/figures/model_comparison.png'))
display(Image('results/figures/data_efficiency.png'))

## 7. Look at the mistakes
High-confidence errors are often ambiguous or mislabelled examples - great material for the README's error analysis.

In [ ]:
import pandas as pd
name = 'distilbert_base_uncased_frac1'
print(pd.read_csv(f'results/{name}/top_confusions.csv').to_string(index=False))
pd.read_csv(f'results/{name}/errors.csv').head(15)

## 8. Download results
Unzip the file into your local repo's `results/` folder, then commit.

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')